In [59]:
import pandas as pd
import numpy as np

# Data Cleaning Tutorial:
# https://www.youtube.com/watch?v=opBAOm851Ng -- ***
# https://www.youtube.com/watch?v=pS6rytmgZKQ -- *****
# https://www.youtube.com/watch?v=sM1mhGVRYI0 -- *****

In [ ]:
# pd.show_versions()

In [ ]:
# pip show pandas

In [61]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from datetime import datetime
import random
from scipy import stats

## CHALLENGE: Update the Codes to clean the Following Generated Codes

In [63]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta

n = 100
# 1. Generate Age with Nulls
age = np.random.normal(30, 10, n)
age[np.random.randint(0, n, 10)] = np.nan

# 2. Generate Income with extreme outliers
income = np.random.normal(50000, 15000, n)
income[np.random.randint(0, n, 5)] *= 10

# 3. Generate Cities with None/Nulls
cities = np.random.choice(['New York', 'Los Angeles', 'Chicago', 'Houston'], n)
cities = cities.astype(object) # Ensure it can hold 'None'
cities[np.random.randint(0, n, 8)] = None

# 4. Generate Dates (Fixed the Null logic)
dates = [datetime.today() - timedelta(days=random.randint(0, 365)) for _ in range(n)]
for idx in np.random.randint(0, n, 5):
    dates[idx] = pd.NaT  # Use NaT (Not a Time) for proper pandas date nulls

# 5. Build the DataFrame
df = pd.DataFrame({
    "Age": age,
    "Income": income,
    "City": cities,
    "Joined": dates
})

# 6. Add the Duplicate row
duplicate_idx = 20
duplicated_row = df.iloc[[duplicate_idx]] # Easier way to grab the whole row
df = pd.concat([df, duplicated_row], ignore_index=True)

# 7. Categorical Notes (Fixed '1' to 'i')
df['notes'] = [
    'good customer' if i % 3 == 0 else 'slow-completer' if i % 4 == 0 else 'frequent payer' 
    for i in range(len(df))
]

print(df.head())

         Age        Income         City                     Joined  \
0        NaN  31508.781804     New York 2026-05-09 14:46:50.498168   
1        NaN  41256.885796     New York 2025-05-19 14:46:50.498228   
2  33.803646  60055.304446  Los Angeles 2025-07-22 14:46:50.498249   
3  34.631418  48490.715663  Los Angeles 2026-04-09 14:46:50.498263   
4  28.595661  49189.007427  Los Angeles 2025-11-04 14:46:50.498275   

            notes  
0   good customer  
1  frequent payer  
2  frequent payer  
3   good customer  
4  slow-completer  


### Production Cleaning

In [52]:
# 1. DATA IMPORT
df = pd.read_csv('anaphylaxis.csv')
df.head()


,ID_Number,Staff_Group,Role,Last_access,Course_Title,Enrolment_Date,Completed,Completion_Date
0,78441.0,Additional Clinical Services,Health Care Support Worker,11/03/2026,Anaphylaxis v4.0,13/03/2025,No,NaN
1,81450.0,Additional Clinical Services,Healthcare Assistant,16/03/2026,Anaphylaxis v4.0,13/01/2026,No,NaN
2,95333.0,Additional Clinical Services,Health Care Support Worker,15/04/2026,Anaphylaxis v4.0,10/12/2025,Yes,10/12/2025
3,93038.0,Nursing and Midwifery Registered,Staff Nurse,07/04/2026,Anaphylaxis v4.0,23/05/2025,Yes,23/05/2025
4,80380.0,Additional Clinical Services,Nursing Associate,14/04/2026,Anaphylaxis v4.0,26/08/2025,Yes,27/08/2025


In [45]:
# df.describe()

In [ ]:
# Creating "Cleaning Functions" that handle text consistency, date conversion, and nulls automatically:

In [64]:
# 2. DEFINING THE "WASH & DRY" FUNCTION
def wash_and_dry(input_df):
    """
    Standardizes the Anaphyl Pipeline datasets for production.
    Ensures lowercase headers, fills nulls, and converts dates.
    """
    # Create a fresh copy to preserve the original 'df' for the briefing comparison
    df_cleaned = input_df.copy() 
    
    # Text Consistency: Normalize column headers
    df_cleaned.columns = [col.lower().replace(' ', '_') for col in df_cleaned.columns]
    
    # Null Management: Identifying 'slow-completers'
    # Note: We check for 'status' in lowercase because the step above renamed it
    if 'status' in df_cleaned.columns:
        df_cleaned['status'] = df_cleaned['status'].fillna('slow-completers')
    
    # Date Conversion
    if 'start_time' in df_cleaned.columns:
        df_cleaned['start_time'] = pd.to_datetime(df_cleaned['start_time'])
    
    return df_cleaned

In [65]:
# 3. SENIOR ANALYST VERIFICATION STEP
print("--- [REPORT] DATA BEFORE CLEANING ---")
print(df.info()) 

# EXECUTION: This is the critical step to create the final data
df_final = wash_and_dry(df)

print("\n--- [REPORT] DATA AFTER CLEANING ---")
print(df_final.info())
print("\n--- SAMPLE OF CLEANED DATA ---")
print(df_final.head())

# Reproducibility standard for subsequent modeling
random_state = 42

--- [REPORT] DATA BEFORE CLEANING ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 101 entries, 0 to 100
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   Age     92 non-null     float64       
 1   Income  101 non-null    float64       
 2   City    93 non-null     object        
 3   Joined  96 non-null     datetime64[ns]
 4   notes   101 non-null    object        
dtypes: datetime64[ns](1), float64(2), object(2)
memory usage: 4.1+ KB
None

--- [REPORT] DATA AFTER CLEANING ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 101 entries, 0 to 100
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   age     92 non-null     float64       
 1   income  101 non-null    float64       
 2   city    93 non-null     object        
 3   joined  96 non-null     datetime64[ns]
 4   notes   101 non-null    object        
dtypes: datetime64[ns](1

#### The Date Conversion Gap (Memory & Logic)

In [ ]:
# The Date Conversion Gap (Memory & Logic)

import pandas as pd

def wash_and_dry_moodle(input_df):
    df_cleaned = input_df.copy()
    
    # 1. Normalize Headers
    df_cleaned.columns = [col.lower().replace(' ', '_') for col in df_cleaned.columns]
    
    # 2. Date Conversion (Targeting your specific columns)
    date_cols = ['last_access', 'enrolment_date', 'completion_date']
    for col in date_cols:
        if col in df_cleaned.columns:
            df_cleaned[col] = pd.to_datetime(df_cleaned[col], dayfirst=True)
    
    # 3. Terminology Harmonization
    # Mapping 'No' to 'slow-completers' to match your project standards
    if 'completed' in df_cleaned.columns:
        df_cleaned['completed'] = df_cleaned['completed'].replace({'No': 'slow-completers'})
        
    return df_cleaned

# Execute the updated wash
df_final = wash_and_dry_moodle(df)

# Briefing Verification
print("--- UPDATED REPORT: DTYPES ---")
print(df_final[['enrolment_date', 'completion_date']].dtypes)
print("\n--- TARGET GROUP CHECK ---")
print(df_final['completed'].value_counts())

#### The Executive Insight: Calculating "Velocity"

In [56]:
# Calculate the duration in days for those who finished
# (Only works because of your successful Stage 3 Date Conversion)
df_final['days_to_complete'] = (df_final['completion_date'] - df_final['enrolment_date']).dt.days

# Get the average for the 'Yes' group
avg_days = df_final[df_final['completed'] == 'Yes']['days_to_complete'].mean()

print(f"Benchmark: On average, successful staff complete the course in {avg_days:.1f} days.")

Benchmark: On average, successful staff complete the course in 9.3 days.


In [ ]:
# ALL IN ONE BLOCK:

import pandas as pd

# 1. DATA IMPORT
df = pd.read_csv('anaphylaxis.csv')

# 2. DEFINING THE "WASH & DRY" FUNCTION
def wash_and_dry(input_df):
    """
    Standardizes the Anaphyl Pipeline datasets for production.
    Ensures lowercase headers, fills nulls, and converts dates.
    """
    # Create a fresh copy to preserve the original 'df' for the briefing comparison
    df_cleaned = input_df.copy() 
    
    # Text Consistency: Normalize column headers
    df_cleaned.columns = [col.lower().replace(' ', '_') for col in df_cleaned.columns]
    
    # Null Management: Identifying 'slow-completers'
    # Note: We check for 'status' in lowercase because the step above renamed it
    if 'status' in df_cleaned.columns:
        df_cleaned['status'] = df_cleaned['status'].fillna('slow-completers')
    
    # Date Conversion
    if 'start_time' in df_cleaned.columns:
        df_cleaned['start_time'] = pd.to_datetime(df_cleaned['start_time'])
    
    return df_cleaned

# 3. SENIOR ANALYST VERIFICATION STEP
print("--- [REPORT] DATA BEFORE CLEANING ---")
print(df.info()) 

# EXECUTION: This is the critical step to create the final data
df_final = wash_and_dry(df)

print("\n--- [REPORT] DATA AFTER CLEANING ---")
print(df_final.info())
print("\n--- SAMPLE OF CLEANED DATA ---")
print(df_final.head())

# Reproducibility standard for subsequent modeling
random_state = 42